# 베이스라인 모델 3종 비교 — XGBoost / LightGBM / CatBoost

목적: 정교한 피처 엔지니어링 전에, **제공된 원본 + asof_* 피처만으로** 세 부스팅 모델의 기본 성능을 비교해서 출발점을 잡는다.

**검증 전략**: 시즌별 `control_success` 비율이 2019(0.565)→2024(0.486)로 꾸준히 하락하는 추세가 있어 랜덤 분할은 미래 정보 누수 위험이 있음.
→ **2019~2023 학습 / 2024 검증**으로 시간 기준 홀드아웃.

**피처**: `pitcher_id`/`batter_id`(고카디널리티 raw ID)는 일단 제외 — 이미 `asof_pitcher_*`/`asof_batter_*`에 이력이 요약되어 있음. 나머지 원본 컬럼 + asof_* 전부 사용, 결측치는 각 라이브러리의 네이티브 처리에 맡김(별도 imputation 안 함).


In [1]:
import time
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, log_loss

pd.set_option("display.max_columns", 60)

DATA_DIR = "../data"
ID_COL = "row_id"
TARGET_COL = "control_success"

train_raw = pd.read_csv(f"{DATA_DIR}/train.csv", encoding="utf-8-sig")
print(train_raw.shape)


(1475092, 49)


## 1. 피처 구성 & 시간 기준 분할

In [2]:
CAT_COLS = ["top_bottom", "game_type", "base_state", "pitcher_hand", "batter_hand",
            "pitcher_team_id", "batter_team_id"]
DROP_COLS = [ID_COL, TARGET_COL, "pitcher_id", "batter_id"]
FEATURE_COLS = [c for c in train_raw.columns if c not in DROP_COLS]

print(f"피처 수: {len(FEATURE_COLS)}")
print(f"범주형: {CAT_COLS}")


피처 수: 45
범주형: ['top_bottom', 'game_type', 'base_state', 'pitcher_hand', 'batter_hand', 'pitcher_team_id', 'batter_team_id']


In [3]:
y = train_raw[TARGET_COL]
train_mask = train_raw["season"] < 2024

# xgboost / lightgbm 용 (category dtype)
X_tree = train_raw[FEATURE_COLS].copy()
for c in CAT_COLS:
    X_tree[c] = X_tree[c].astype(str).astype("category")

# catboost 용 (범주형은 문자열로)
X_cb = train_raw[FEATURE_COLS].copy()
for c in CAT_COLS:
    X_cb[c] = X_cb[c].astype(str)

X_train_tree, X_valid_tree = X_tree[train_mask], X_tree[~train_mask]
X_train_cb, X_valid_cb = X_cb[train_mask], X_cb[~train_mask]
y_train, y_valid = y[train_mask], y[~train_mask]

print(f"train: {X_train_tree.shape}, valid: {X_valid_tree.shape}")
print(f"train 성공률: {y_train.mean():.4f}, valid 성공률: {y_valid.mean():.4f}")


train: (1221585, 45), valid: (253507, 45)
train 성공률: 0.5316, valid 성공률: 0.4861


## 2. XGBoost

In [4]:
import xgboost as xgb

t0 = time.time()
xgb_model = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    tree_method="hist", enable_categorical=True,
    eval_metric="logloss", early_stopping_rounds=50,
    n_jobs=-1, random_state=42,
)
xgb_model.fit(X_train_tree, y_train, eval_set=[(X_valid_tree, y_valid)], verbose=False)
xgb_time = time.time() - t0

xgb_pred = xgb_model.predict_proba(X_valid_tree)[:, 1]
xgb_auc = roc_auc_score(y_valid, xgb_pred)
xgb_ll = log_loss(y_valid, xgb_pred)
print(f"XGBoost — AUC={xgb_auc:.5f}  LogLoss={xgb_ll:.5f}  best_iter={xgb_model.best_iteration}  time={xgb_time:.1f}s")


XGBoost — AUC=0.54369  LogLoss=0.68990  best_iter=96  time=10.7s


## 3. LightGBM

In [5]:
import lightgbm as lgb

t0 = time.time()
lgb_model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1,
)
lgb_model.fit(
    X_train_tree, y_train,
    eval_set=[(X_valid_tree, y_valid)], eval_metric="logloss",
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
lgb_time = time.time() - t0

lgb_pred = lgb_model.predict_proba(X_valid_tree)[:, 1]
lgb_auc = roc_auc_score(y_valid, lgb_pred)
lgb_ll = log_loss(y_valid, lgb_pred)
print(f"LightGBM — AUC={lgb_auc:.5f}  LogLoss={lgb_ll:.5f}  best_iter={lgb_model.best_iteration_}  time={lgb_time:.1f}s")


/Users/piropilho/Desktop/lg_aimers/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM — AUC=0.54432  LogLoss=0.68976  best_iter=113  time=5.1s


## 4. CatBoost

In [6]:
from catboost import CatBoostClassifier

t0 = time.time()
cb_model = CatBoostClassifier(
    iterations=500, learning_rate=0.05, depth=6,
    cat_features=CAT_COLS, eval_metric="Logloss",
    random_seed=42, verbose=False, early_stopping_rounds=50,
)
cb_model.fit(X_train_cb, y_train, eval_set=(X_valid_cb, y_valid))
cb_time = time.time() - t0

cb_pred = cb_model.predict_proba(X_valid_cb)[:, 1]
cb_auc = roc_auc_score(y_valid, cb_pred)
cb_ll = log_loss(y_valid, cb_pred)
print(f"CatBoost — AUC={cb_auc:.5f}  LogLoss={cb_ll:.5f}  best_iter={cb_model.get_best_iteration()}  time={cb_time:.1f}s")


CatBoost — AUC=0.54811  LogLoss=0.68912  best_iter=215  time=65.4s


## 5. 결과 비교

In [7]:
results = pd.DataFrame([
    {"model": "XGBoost", "AUC": xgb_auc, "LogLoss": xgb_ll, "fit_time_s": xgb_time},
    {"model": "LightGBM", "AUC": lgb_auc, "LogLoss": lgb_ll, "fit_time_s": lgb_time},
    {"model": "CatBoost", "AUC": cb_auc, "LogLoss": cb_ll, "fit_time_s": cb_time},
]).sort_values("AUC", ascending=False)
results


,model,AUC,LogLoss,fit_time_s
2,CatBoost,0.548115,0.689116,65.396220
1,LightGBM,0.544320,0.689762,5.123785
0,XGBoost,0.543690,0.689904,10.689406


## 6. 피처 중요도 (모델 간 공통적으로 중요한 피처 확인)

In [8]:
imp = pd.DataFrame({
    "feature": FEATURE_COLS,
    "xgb": xgb_model.feature_importances_,
    "lgb": lgb_model.feature_importances_ / lgb_model.feature_importances_.sum(),
    "cb": cb_model.get_feature_importance() / 100,
}).set_index("feature")
imp["avg_rank"] = imp.rank(ascending=False).mean(axis=1)
imp.sort_values("avg_rank").head(15)


,xgb,lgb,cb,avg_rank
feature,,,,
season,0.089744,0.051385,0.187679,2.333333
asof_pitcher_reverse_rate,0.045243,0.041536,0.051218,4.666667
asof_pitcher_success_rate,0.085494,0.034970,0.082033,5.333333
pitcher_team_id,0.013047,0.071082,0.035494,6.333333
batter_team_id,0.012552,0.069655,0.042016,7.000000
asof_batter_success_rate,0.015683,0.038538,0.024120,7.333333
asof_pitcher_ball_rate,0.014535,0.042820,0.027051,7.333333
asof_pitcher_prev3_game_success_rate,0.017908,0.032401,0.021760,9.000000
asof_pitcher_prev1_game_success_rate,0.014705,0.039109,0.016444,10.000000


## 8. 도메인 피처 엔지니어링 반영

브레인스토밍/EDA에서 확인한 가설들을 명시적 피처로 변환:
- **카운트 압박**: `count_code`(볼-스트라이크 조합, 범주형), `full_count`, `three_ball`, `two_strike` — full count 등 소수 조합의 강한 신호를 별도 분리 없이도 트리가 바로 활용하게
- **매치업**: `same_hand`, `hand_matchup`
- **점수차/블로아웃**: `score_abs_pitcher`, `blowout`(5점차 이상), `close_game`(1점차 이내)
- **주자 압박**: `runner_scoring_pos`(2·3루 합)
- **최근 폼**: `form_delta1/3/5`(직전 N경기 성공률 - 시즌 성공률), `form_declining` 플래그
- **구종 다양성**: `pitchmix_entropy`, `pitchmix_max` (fastball/breaking/offspeed 비율의 엔트로피 — 단일 컬럼 3개로는 트리가 근사하기 어려운 비선형 조합)
- **투수 커맨드 종합 지표**: `command_quality`(성공률 - 반대성 - 가운데), `strike_ball_gap`
- **위기 결합 스코어**: `stress_score` = full_count + blowout + form_declining 합


In [9]:
def engineer_features(df):
    x = df.copy()
    x["count_code"] = x["balls_before"].astype(str) + "-" + x["strikes_before"].astype(str)
    x["full_count"] = ((x["balls_before"] == 3) & (x["strikes_before"] == 2)).astype("int8")
    x["three_ball"] = (x["balls_before"] == 3).astype("int8")
    x["two_strike"] = (x["strikes_before"] == 2).astype("int8")

    x["same_hand"] = (x["pitcher_hand"] == x["batter_hand"]).astype("int8")
    x["hand_matchup"] = x["pitcher_hand"].astype(str) + "-" + x["batter_hand"].astype(str)

    x["score_abs_pitcher"] = x["score_diff_pitcher_team"].abs()
    x["blowout"] = (x["score_abs_pitcher"] >= 5).astype("int8")
    x["close_game"] = (x["score_abs_pitcher"] <= 1).astype("int8")

    x["runner_scoring_pos"] = x["runner_on_2b"] + x["runner_on_3b"]
    x["late_inning"] = (x["inning"] >= 7).astype("int8")

    x["form_delta1"] = x["asof_pitcher_prev1_game_success_rate"] - x["asof_pitcher_success_rate"]
    x["form_delta3"] = x["asof_pitcher_prev3_game_success_rate"] - x["asof_pitcher_success_rate"]
    x["form_delta5"] = x["asof_pitcher_prev5_game_success_rate"] - x["asof_pitcher_success_rate"]
    x["form_declining"] = (x["form_delta1"] < -0.05).astype("int8")

    mix = x[["asof_pitcher_fastball_rate", "asof_pitcher_breaking_rate", "asof_pitcher_offspeed_rate"]].clip(1e-6, 1).fillna(1 / 3)
    x["pitchmix_entropy"] = -(mix * np.log(mix)).sum(axis=1)
    x["pitchmix_max"] = mix.max(axis=1)

    x["command_quality"] = x["asof_pitcher_success_rate"] - x["asof_pitcher_reverse_rate"] - x["asof_pitcher_middle_rate"]
    x["strike_ball_gap"] = x["asof_pitcher_strike_rate"] - x["asof_pitcher_ball_rate"]

    x["stress_score"] = x["full_count"] + x["blowout"] + x["form_declining"]
    return x


ENG_CAT_COLS = CAT_COLS + ["count_code", "hand_matchup"]

train_eng = engineer_features(train_raw)
ENG_FEATURE_COLS = [c for c in train_eng.columns if c not in DROP_COLS]
print(f"피처 수: {len(FEATURE_COLS)} -> {len(ENG_FEATURE_COLS)}")


피처 수: 45 -> 65


In [10]:
X_tree2 = train_eng[ENG_FEATURE_COLS].copy()
for c in ENG_CAT_COLS:
    X_tree2[c] = X_tree2[c].astype(str).astype("category")

X_cb2 = train_eng[ENG_FEATURE_COLS].copy()
for c in ENG_CAT_COLS:
    X_cb2[c] = X_cb2[c].astype(str)

X_train_tree2, X_valid_tree2 = X_tree2[train_mask], X_tree2[~train_mask]
X_train_cb2, X_valid_cb2 = X_cb2[train_mask], X_cb2[~train_mask]

print(X_train_tree2.shape, X_valid_tree2.shape)


(1221585, 65) (253507, 65)


In [11]:
t0 = time.time()
xgb_model2 = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    tree_method="hist", enable_categorical=True,
    eval_metric="logloss", early_stopping_rounds=50,
    n_jobs=-1, random_state=42,
)
xgb_model2.fit(X_train_tree2, y_train, eval_set=[(X_valid_tree2, y_valid)], verbose=False)
xgb_time2 = time.time() - t0
xgb_pred2 = xgb_model2.predict_proba(X_valid_tree2)[:, 1]
xgb_auc2 = roc_auc_score(y_valid, xgb_pred2)
xgb_ll2 = log_loss(y_valid, xgb_pred2)
print(f"XGBoost(eng) — AUC={xgb_auc2:.5f}  LogLoss={xgb_ll2:.5f}  time={xgb_time2:.1f}s")


XGBoost(eng) — AUC=0.54813  LogLoss=0.68925  time=20.9s


In [12]:
t0 = time.time()
lgb_model2 = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1,
)
lgb_model2.fit(
    X_train_tree2, y_train,
    eval_set=[(X_valid_tree2, y_valid)], eval_metric="logloss",
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
lgb_time2 = time.time() - t0
lgb_pred2 = lgb_model2.predict_proba(X_valid_tree2)[:, 1]
lgb_auc2 = roc_auc_score(y_valid, lgb_pred2)
lgb_ll2 = log_loss(y_valid, lgb_pred2)
print(f"LightGBM(eng) — AUC={lgb_auc2:.5f}  LogLoss={lgb_ll2:.5f}  time={lgb_time2:.1f}s")


/Users/piropilho/Desktop/lg_aimers/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM(eng) — AUC=0.54826  LogLoss=0.68924  time=12.2s


In [13]:
t0 = time.time()
cb_model2 = CatBoostClassifier(
    iterations=500, learning_rate=0.05, depth=6,
    cat_features=ENG_CAT_COLS, eval_metric="Logloss",
    random_seed=42, verbose=False, early_stopping_rounds=50,
)
cb_model2.fit(X_train_cb2, y_train, eval_set=(X_valid_cb2, y_valid))
cb_time2 = time.time() - t0
cb_pred2 = cb_model2.predict_proba(X_valid_cb2)[:, 1]
cb_auc2 = roc_auc_score(y_valid, cb_pred2)
cb_ll2 = log_loss(y_valid, cb_pred2)
print(f"CatBoost(eng) — AUC={cb_auc2:.5f}  LogLoss={cb_ll2:.5f}  time={cb_time2:.1f}s")


CatBoost(eng) — AUC=0.54968  LogLoss=0.68888  time=109.0s


## 9. 결과 비교 — 베이스라인 vs 피처 엔지니어링

In [14]:
results2 = pd.DataFrame([
    {"model": "XGBoost", "stage": "baseline", "AUC": xgb_auc, "LogLoss": xgb_ll},
    {"model": "XGBoost", "stage": "engineered", "AUC": xgb_auc2, "LogLoss": xgb_ll2},
    {"model": "LightGBM", "stage": "baseline", "AUC": lgb_auc, "LogLoss": lgb_ll},
    {"model": "LightGBM", "stage": "engineered", "AUC": lgb_auc2, "LogLoss": lgb_ll2},
    {"model": "CatBoost", "stage": "baseline", "AUC": cb_auc, "LogLoss": cb_ll},
    {"model": "CatBoost", "stage": "engineered", "AUC": cb_auc2, "LogLoss": cb_ll2},
])
pivot = results2.pivot(index="model", columns="stage", values="AUC")
pivot["delta"] = pivot["engineered"] - pivot["baseline"]
pivot.sort_values("engineered", ascending=False)


stage,baseline,engineered,delta
model,,,
CatBoost,0.548115,0.549678,0.001563
LightGBM,0.544320,0.548262,0.003942
XGBoost,0.543690,0.548127,0.004437


## 10. 피처 중요도 (엔지니어링 반영 후, CatBoost 기준 — 가장 성능 좋았던 모델)

In [15]:
imp2 = pd.Series(cb_model2.get_feature_importance(), index=ENG_FEATURE_COLS).sort_values(ascending=False)
imp2.head(20)


game_type                               29.997236
season                                  18.750922
asof_pitcher_success_rate                6.545506
batter_team_id                           4.224111
command_quality                          4.093574
pitcher_team_id                          3.413596
hand_matchup                             3.199359
asof_pitcher_reverse_rate                2.802401
asof_batter_success_rate                 2.206057
asof_pitcher_ball_rate                   1.691931
asof_pitcher_prev5_game_success_rate     1.660045
asof_pitcher_prev3_game_success_rate     1.610865
game_month                               1.574291
strike_ball_gap                          1.416058
asof_pitcher_n                           1.276440
asof_pitcher_prev1_game_success_rate     1.221687
asof_pitcher_pitchmix_n                  1.018389
form_delta5                              0.968054
asof_pitcher_offspeed_rate               0.844402
balls_before                             0.803900


## 7. 다음 스텝 메모

- (결과 보고 어떤 모델/피처 방향으로 이어갈지 정리)
